In [1]:
import os
import re
import urllib.request

import matplotlib as mlp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from scipy.optimize import minimize

FIGSIZE = (20, 16)

# Nombres de países

In [2]:
#
#  Nombres unicos de paises en los datasets
#
country_names_dict = {
    "côte_d'ivoire": "cote_d'ivoire",
    "brunei_": "brunei",
    "cabo_verde": "cape_verde",
    "china__hong_kong": "hong_kong",
    "côte_divoire": "cote_d'ivoire",
    "curaçao": "curacao",
    "czech_republic_(czechia)": "czech_republic",
    "czechia": "czech_republic",
    "democratic_republic_of_congo": "dr_congo",
    "denmark__greenland": "greenland",
    "eswatini": "swaziland",
    "france__french_guiana": "french_guiana",
    "france__french_polynesia": "french_polynesia",
    "france__guadeloupe": "guadeloupe",
    "france__martinique": "martinique",
    "france__mayotte": "mayotte",
    "france__new_caledonia": "new_caledonia",
    "france__reunion": "reunion",
    "france__saint_barthelemy": "saint_barthelemy",
    "france__saint_pierre_and_miquelon": "pierre_and_miquelon",
    "france__st_martin": "saint_martin",
    "korea,_south": "south_korea",
    "macedonia": "north_macedonia",
    "myanmar": "burma",
    "netherlands__aruba": "aruba",
    "netherlands__curacao": "curacao",
    "netherlands__sint_maarten": "sint_maarten",
    "réunion": "reunion",
    "saint_pierre_andamp;_miquelon": "saint_pierre_and_miquelon",
    "sao_tome_andamp;_principe": "sao_tome_and_principe",
    "sint_maarten_(dutch_part)": "sint_maarten",
    "state_of_palestine": "palestine",
    "taiwan*": "taiwan",
    "timor-leste": "timor",
    "turks_and_caicos_islands": "turks_and_caicos",
    "united_kingdom__anguilla": "anguilla",
    "united_kingdom__bermuda": "bermuda",
    "united_kingdom__british_virgin_islands": "british_virgin_islands",
    "united_kingdom__cayman_islands": "cayman_islands",
    "united_kingdom__channel_islands": "channel_islands",
    "united_kingdom__falkland_islands_(malvinas)": "falkland_islands",
    "united_kingdom__gibraltar": "gibraltar",
    "united_kingdom__isle_of_man": "isle_of_man",
    "united_kingdom__montserrat": "montserrat",
    "united_kingdom__turks_and_caicos_islands": "turks_and_caicos",
    "us": "united_states",
    "st._vincent_andamp;_grenadines": "saint_vincent_and_the_grenadines",
    "united_states_virgin_islands": "us_virgin_islands",
    "u.s._virgin_islands": "us_virgin_islands",
    "saint_kitts_andamp;_nevis": "saint_kitts_and_nevis",
}

In [14]:
#
#  Nombres que se pueden borrar
#
names_to_pop = [
    "ms_zaandam",
    "holy_see",
    "international",
    "vatican",
    'diamond_princess',
    "world",
]

# Datos Genéricos de Países (solo se requiere una vez)

Se corre para generar el archivo **world_data.csv**.

www.worldometers.info/world-population/population-by-country/

https://ourworldindata.org

https://github.com/owid/covid-19-data/tree/master/public/data

In [15]:
#
# Captura y descarga de datos
#
def create_world_data_csv():
    """Captures world data population data.
    """
    url = "https://www.worldometers.info/world-population/population-by-country/"
    response = requests.get(url)
    x = BeautifulSoup(response.text, "html.parser")
    x = x.findAll("td")
    x = [repr(a) for a in x]
    x = [
        a.replace("<td>", "")
        .replace("</td>", "")
        .replace(",", "")
        .replace(" %", "")
        .replace('<td style="font-weight: bold;">', "")
        .replace('<td style="font-weight: bold; font-size:15px; text-align:left">', "")
        .replace("</a>", "")
        for a in x
    ]
    x = [a[a.find(">") + 1 :] for a in x]
    #
    country = [a.replace(" ", "_").replace("&", "and").lower() for a in x[1::12]]
    population = [float(a) for a in x[2::12]]
    density = [float(a) for a in x[5::12]]
    land_area = [float(a) for a in x[6::12]]
    med_age = [float(a) if a != "N.A." else None for a in x[9::12]]
    urb_pop = [float(a) / 100 if a != "N.A." else None for a in x[10::12]]
    #
    world_data = pd.DataFrame(
        {
            "country": country,
            "population": population,
            "density": density,
            "land_area": land_area,
            "med_age": med_age,
            "urb_pop": urb_pop,
        }
    )
    #
    # actual <-- nuevo
    #
    replace_list = list(country_names_dict.keys())
    for key in country_names_dict:
        old_name = key
        new_name = country_names_dict[key]
        world_data.country = world_data.country.map(
            lambda x: x.replace(old_name, new_name) if x in replace_list else x
        )
    #
    world_data.to_csv("world_data.csv", index=False)


## create_world_data_csv()

In [16]:
#
# Agrega datos de OWiD al archivo world_data.csv
#
def adds_OWiD_to_world_data_csv():
    OWiD = pd.read_csv(
        "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/"
        + "owid-covid-data.csv"
    )
    OWiD.location = OWiD.location.map(lambda x: x.lower().replace(" ", "_"))
    OWiD.location = OWiD.location.map(
        lambda x: country_names_dict[x] if x in country_names_dict.keys() else x
    )
    last_date = sorted(set(OWiD.date))[-1]
    last_data = OWiD[OWiD.date == last_date]
    world_data = pd.read_csv("world_data.csv")
    world_data = world_data.set_index("country")
    #
    columns = [
        "aged_65_older",
        "aged_70_older",
        "cvd_death_rate",
        "diabetes_prevalence",
        "extreme_poverty",
        "female_smokers",
        "gdp_per_capita",
        "handwashing_facilities",
        "hospital_beds_per_100k",
        "male_smokers",
        "median_age",
        "stringency_index",
    ]
    for column in columns:
        world_data[column] = None
    #
    for _, row in OWiD.iterrows():
        if row.location in list(world_data.index):
            for column in columns:
                world_data.at[row.location, column] = row[column]
    world_data = world_data.reset_index()
    world_data.to_csv("world_data.csv", index=False)


## adds_OWiD_to_world_data_csv()

# Lectura de Datos --- John Hopkins University

https://github.com/CSSEGISandData/COVID-19

In [17]:
def download_jhu_data(filename):

    df = pd.read_csv(
        "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/"
        + filename
    )
    df["Province/State"] = df["Province/State"].map(
        lambda x: "" if pd.isna(x) else "__" + x
    )
    df = df.assign(country=df["Country/Region"] + df["Province/State"])
    df.pop("Lat")
    df.pop("Long")
    df.pop("Province/State")
    df.pop("Country/Region")
    df = df.set_index("country")
    df = df.transpose()
    df.columns = [x.lower().replace(" ", "_") for x in df.columns]
    df = df.reset_index()
    df = df.rename(columns={"index": "date"})
    #  df.date = df.date.map(lambda x: x.split('/'))
    #  df.date = df.date.map(lambda x: "0" + x if len(x) == 7 else x)
    # df.date = df.date.map(lambda x: "20" + x[-2:] + "-" + x[:2] + "-" + x[3:5])
    #
    df["canada"] = 0
    for col in df.columns:
        if "canada__" in col:
            df.canada = df.canada + df[col]
            df.pop(col)

    df["australia"] = 0
    for col in df.columns:
        if "australia__" in col:
            df.australia = df.australia + df[col]
            df.pop(col)

    df["china"] = 0
    for col in df.columns:
        if "china__" in col:
            df.china = df.china + df[col]
            df.pop(col)

    df["congo"] = 0
    for col in df.columns:
        if "congo_" in col:
            df.congo = df.congo + df[col]
            df.pop(col)
    #
    df = df.rename(columns=country_names_dict)

    for term in names_to_pop:
        if term in list(df.columns):
            df.pop(term)

    #
    return df


CONFIRMED = download_jhu_data("time_series_covid19_confirmed_global.csv")
DEATHS = download_jhu_data("time_series_covid19_deaths_global.csv")
RECOVERED = download_jhu_data("time_series_covid19_recovered_global.csv")

CONFIRMED = CONFIRMED.set_index("date")
DEATHS = DEATHS.set_index("date")
RECOVERED = RECOVERED.set_index("date")

ACTIVE = CONFIRMED - DEATHS - RECOVERED

NEW_CASES = CONFIRMED - CONFIRMED.shift(periods=1, fill_value=0)
NEW_DEATHS = DEATHS - DEATHS.shift(periods=1, fill_value=0)
NEW_RECOVERED = RECOVERED - RECOVERED.shift(periods=1, fill_value=0)

# Tests reportados en Our World in Data.

In [18]:
def download_OWiD_data():
    #
    def transpose_data(column):
        data = OWiD[["date", "location", column]]
        data = pd.pivot_table(data, columns="location", index="date")
        data.columns = list(data.columns.droplevel())
        data = data.reset_index()
        data = data[data.date.map(lambda x: x in list(CONFIRMED.index))]
        return data

    #
    OWiD = pd.read_csv(
        "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/"
        + "owid-covid-data.csv"
    )
    OWiD.location = OWiD.location.map(lambda x: x.lower().replace(" ", "_"))
    OWiD.location = OWiD.location.map(
        lambda x: country_names_dict[x] if x in country_names_dict.keys() else x
    )
    #
    # OWiD = OWiD.rename(columns={"date": "fecha"})
    #
    total_tests = transpose_data("total_tests")
    new_tests = transpose_data("new_tests")
    #
    total_tests = total_tests.set_index("date")
    new_tests = new_tests.set_index("date")
    return total_tests, new_tests


ACTIVE = CONFIRMED - DEATHS - RECOVERED

NEW_CASES = CONFIRMED - CONFIRMED.shift(periods=1, fill_value=0)
NEW_DEATHS = DEATHS - DEATHS.shift(periods=1, fill_value=0)
NEW_RECOVERED = RECOVERED - RECOVERED.shift(periods=1, fill_value=0)
NEW_TESTS = download_OWiD_data()

# Tasas

In [19]:
TASA_CONTAGIO = NEW_CASES / ACTIVE.shift(periods=1, fill_value=1)
TASA_MUERTE = NEW_DEATHS / ACTIVE.shift(periods=1, fill_value=1)
TASA_RECUPERACION = NEW_RECOVERED / ACTIVE.shift(periods=1, fill_value=1)

# Carga de world_data.csv

In [20]:
world_data = pd.read_csv("world_data.csv")

# ANÁLISIS BÁSICO

# Paises con más casos (Poblacion > 10.000.000)

In [21]:
def total_casos_ordenados():
    x = CONFIRMED.tail(1)
    x = x.transpose()
    x.columns = ["total_cases"]
    x = x.sort_values(by="total_cases", ascending=False)
    return x


total_casos_ordenados().head(15)

,total_cases
united_states,2422299
brazil,1228114
russia,613148
india,490401
united_kingdom,307980
peru,268602
chile,259064
spain,247486
italy,239706
iran,215096


# Paises con más casos por 100k habitantes

In [22]:
def total_casos_por_100k():
    x = CONFIRMED.tail(1)
    x = x.transpose()
    x.columns = ["total_cases"]
    x = x.reset_index()
    x.columns = ["country", "total_cases"]
    x = x[x.country.map(lambda w: w in list(world_data.country))]
    population = {b: a for a, b in zip(world_data.population, world_data.country)}
    x["population"] = x.country.map(lambda w: population[w])
    x = x.assign(total_cases_100k=x.total_cases / x.population * 100000)
    x = x.sort_values(by="total_cases_100k", ascending=False)
    x = x[x.population > 10000000]
    x = x[["country", "total_cases_100k", "total_cases"]]
    return x


total_casos_por_100k().head(20)

,country,total_cases_100k,total_cases
29,chile,1355.206508,259064
125,peru,814.640269,268602
169,united_states,731.806526,2422299
149,sweden,632.620295,63890
20,brazil,577.774449,1228114
145,spain,529.327719,247486
15,belgium,526.393309,61007
136,saudi_arabia,490.146586,170639
167,united_kingdom,453.672260,307980
131,russia,420.152986,613148


# Paises finalizando la pandemia

In [ ]:
grupo1 = list(total_casos_ordenados().head(60).index)
grupo2 = list(total_casos_por_100k().head(60).country)
countries = sorted(set(grupo1 + grupo2))
countries

casos_activos_hoy = ACTIVE[countries].tail(1)
casos_activos_hoy = casos_activos_hoy.transpose()
casos_activos_hoy.columns = ["casos_activos"]
casos_activos_hoy = casos_activos_hoy.casos_activos
casos_activos_max = ACTIVE[countries].max()

pico = casos_activos_hoy / casos_activos_max
pico = pico[pico < 0.6]
pico.sort_values()

In [ ]:
countries = list(pico.index)
## countries.remove('china')
countries = countries[:20]

In [23]:
countries = total_casos_ordenados().index

# Casos por cada 100k habitantes

In [ ]:
#
#  Elimina los paises que no pertencen al grupo de interes
#
CONFIRMED = CONFIRMED[countries]
DEATHS = DEATHS[countries]
RECOVERED = RECOVERED[countries]
ACTIVE = ACTIVE[countries]
NEW_CASES = NEW_CASES[countries]
NEW_DEATHS = NEW_DEATHS[countries]
NEW_RECOVERED = NEW_RECOVERED[countries]

In [28]:
def by100k(df):
    result = df.copy()
    result = result.astype(float)
    population = {c: p for c, p in zip(world_data.country, world_data.population)}
    for country in result.columns:
        if country in population.keys():
            result[country] = result[country].map(
                lambda x: round(x / population[country] * 100000, 1)
            )
    return result


CONFIRMED_100K = by100k(CONFIRMED)
DEATHS_100K = by100k(DEATHS)
RECOVERED_100K = by100k(RECOVERED)
ACTIVE_100K = by100k(ACTIVE)
NEW_CASES_100K = by100k(NEW_CASES)
NEW_DEATHS_100K = by100k(NEW_DEATHS)
NEW_RECOVERED_100K = by100k(NEW_RECOVERED)

# Gráficos generales

In [ ]:
def format_plot(country):
    #  plt.gca().get_yaxis().set_major_formatter(
    #      mlp.ticker.FuncFormatter(lambda x, p: str(int(x)) + "k")
    #  )
    plt.gca().spines["top"].set_visible(False)
    plt.gca().spines["right"].set_visible(False)
    plt.gca().spines["left"].set_visible(False)
    plt.gca().spines["bottom"].set_visible(False)
    plt.text(0, 0.95 * plt.ylim()[1], country)
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(5))

### Confirmados

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(CONFIRMED_100K[country])
    format_plot(country)

### Nuevos casos

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(NEW_CASES_100K[country])
    format_plot(country)

### Casos activos

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(ACTIVE_100K[country])
    format_plot(country)

### Recuperados

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(RECOVERED_100K[country])
    format_plot(country)

### Muertos

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(DEATHS_100K[country])
    format_plot(country)

In [ ]:
from numpy import power, prod


def media_tasas(x):
    n = len(x)
    x = [u + 1 for u in x]
    x = prod(x)
    x = power(x, 1.0 / n) - 1.0
    return x

In [ ]:
def compute_moving_average(df, time=7):

    df = df.copy()
    df = df.applymap(lambda x: 0.0 if pd.isna(x) else x)
    result = 1 + df.copy()
    for period in range(1, time):
        result = result * (1 + df.shift(periods=period, fill_value=0))
    result = result.applymap(
        lambda x: np.power(x, 1.0 / time) - 1.0 if x is not None else x
    )
    return result


TASA_CONTAGIO_MA5 = compute_moving_average(TASA_CONTAGIO, time=5)
TASA_MUERTE_MA5 = compute_moving_average(TASA_MUERTE, time=5)
TASA_RECUPERACION_MA5 = compute_moving_average(TASA_RECUPERACION, time=5)

### Tasa de contagio

In [ ]:
plt.figure(figsize=(FIGSIZE))
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(TASA_CONTAGIO[country], alpha=0.5)
    plt.plot(TASA_CONTAGIO_MA5[country], "-r", linewidth=2)
    format_plot(country)

### Tasa de muerte

In [ ]:
plt.figure(figsize=(FIGSIZE))
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(TASA_MUERTE[country], alpha=0.5)
    plt.plot(TASA_MUERTE_MA5[country], "-r", linewidth=2)
    format_plot(country)

### Tasa de recuperacion

In [ ]:
plt.figure(figsize=(FIGSIZE))
for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    plt.plot(TASA_RECUPERACION[country], alpha=0.5)
    plt.plot(TASA_RECUPERACION_MA5[country], "-r", linewidth=2)
    format_plot(country)

# INDICADORES DE CARACTERIZACION

In [ ]:
df = pd.DataFrame()
df["confirmed_100k"] = list(CONFIRMED_100K.max())
df.index = CONFIRMED_100K.columns
df

In [ ]:
#
# Máximo cantidad de casos activos (pico de la curva)
#
df["max_active_100k"] = 0.0
for country in countries:
    df.at[country, "max_active_100k"] = ACTIVE_100K[country].max()
df

In [ ]:
#
# Máximo número de casos diario para t >= 100 casos totales
#
df["max_new_cases_100k"] = 0.0
for country in countries:
    df.at[country, "max_new_cases_100k"] = NEW_CASES_100K[country][
        CONFIRMED[country] > 100
    ].max()
df

In [ ]:
#
# Maxima tasa diaria de contagio para
# para t >= 100 casos totales
#
df["max_tasa_contagio"] = 0.0
for country in countries:
    df.at[country, "max_tasa_contagio"] = TASA_CONTAGIO[country][
        CONFIRMED[country] > 100
    ].max()
df

In [ ]:
#
# Dias al pico de activos
#
df["days_to_max_active"] = 0

ACTIVE_100K["days"] = range(len(ACTIVE_100K))

for country in countries:
    first_day = sum(CONFIRMED[country].map(lambda x: x < 100))
    pike_day = int(
        ACTIVE_100K["days"][
            ACTIVE_100K[country].map(lambda x: x == ACTIVE_100K[country].max())
        ].head(1)
    )
    df.at[country, "days_to_max_active"] = pike_day - first_day

ACTIVE_100K.pop("days")
df

In [ ]:
#
# Dias entre el 20% y 80% infectados
#
CONFIRMED["days"] = range(len(CONFIRMED))
df["days_20_to_80"] = 0

for country in countries:
    first_day = sum(
        CONFIRMED[country].map(lambda x: x < 0.20 * CONFIRMED[country].max())
    )
    last_day = sum(
        CONFIRMED[country].map(lambda x: x < 0.80 * CONFIRMED[country].max())
    )
    df.at[country, "days_20_to_80"] = last_day - first_day

CONFIRMED.pop("days")
df

In [ ]:
world_data = world_data.set_index("country")

columns = [
    "density",
    "land_area",
    "med_age",
    "urb_pop",
    "aged_65_older",
    "aged_70_older",
    "cvd_death_rate",
    "diabetes_prevalence",
    "extreme_poverty",
    "gdp_per_capita",
    "handwashing_facilities",
    "hospital_beds_per_100k",
    "male_smokers",
    "stringency_index",
]

for col in columns:
    df[col] = 0
    for country in countries:
        df.at[country, col] = (
            world_data[col][country]
            if pd.isna(world_data[col][country]) is False
            else None
        )
world_data = world_data.reset_index()
df

In [ ]:
#  df.reset_index()
#  df.to_csv('consolidado.csv')

# MODELO BASS

In [ ]:
def bass_model_Ft(t_max, p, q):
    t = np.array(list(range(t_max)))
    num = 1 - np.exp(-(p + q) * t)
    den = 1 + (p / q) * np.exp(-(p + q) * t)
    return num / den


def bass_model_ft(t_max, p, q):
    t = np.array(list(range(t_max)))
    num = np.power(p + q, 2) / p * np.exp(-(p + q) * t)
    den = np.power(1 + (q / p) * np.exp(-(p + q) * t), 2)
    return num / den

In [ ]:
Ft = bass_model_Ft(t_max=100, p=0.1, q=0.004)
plt.plot(Ft, "-k");

In [ ]:
ft = bass_model_ft(t_max=100, p=0.1, q=0.004)
plt.plot(ft, "-k");

In [ ]:
def compute_bass_model(confirmed):
    #
    def loss(w):
        p = w[0]
        q = w[1]
        Ft = bass_model_Ft(t_max=len(yt), p=p, q=q)
        sse = sum((np.array(yt) - Ft) ** 2)
        return sse
    #
    yt = confirmed[confirmed > 0].tolist()
    w = [0.1, 0.01]
    w = minimize(loss, w).x
    bass = bass_model_Ft(len(yt), w[0], w[1])
    result = [0] * (len(confirmed) - len(yt)) + list(bass)
    return result, w[0], w[1]


bass_data = {}
plt.figure(figsize=FIGSIZE)
BASS_CONFIRMED_100K = CONFIRMED_100K.copy()
BASS_CONFIRMED_100K = BASS_CONFIRMED_100K.applymap(lambda w: 0)
for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    max_country = CONFIRMED_100K[country].max()
    real = 0.98 * CONFIRMED_100K[country] / max_country
    forecast, p, q = compute_bass_model(real)
    bass_data[country] = (p, q)
    forecast = [w * max_country for w in forecast]
    forecast = pd.DataFrame(
        {"real": CONFIRMED_100K[country], "forecast": forecast},
        index=CONFIRMED_100K.index,
    )
    #
    BASS_CONFIRMED_100K[country] = forecast.forecast
    #
    plt.plot(forecast.real, "-k", linewidth=2, alpha=1.0)
    plt.plot(forecast.forecast, "-r", linewidth=5, alpha=0.4)
    plt.text(0, 0.60 * plt.ylim()[1], "p={:5.4f}".format(p))
    plt.text(0, 0.50 * plt.ylim()[1], "q={:5.4f}".format(q))
    format_plot(country)

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    plt.plot(BASS_CONFIRMED_100K[country], '-r', linewidth=3)
    format_plot(country)

In [ ]:
BASS_NEW_CASES_100K = BASS_CONFIRMED_100K - BASS_CONFIRMED_100K.shift(periods=1, fill_value=0)

plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    plt.plot(NEW_CASES_100K[country], alpha=0.5)
    plt.plot(BASS_NEW_CASES_100K[country], '-r', linewidth=3)
    format_plot(country)

In [ ]:
bass_data

# OTRAS CURVAS BASS

In [ ]:
RECOVERED_DEATHS_100K = RECOVERED_100K + DEATHS_100K

plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    plt.plot(RECOVERED_DEATHS_100K[country], '-r', linewidth=3)
    format_plot(country)

In [ ]:
bass_data = {}
plt.figure(figsize=FIGSIZE)
BASS_RECOVERED_DEATHS_100K = CONFIRMED_100K.copy()
BASS_RECOVERED_DEATHS_100K = BASS_RECOVERED_DEATHS_100K.applymap(lambda w: 0)

for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    max_country = RECOVERED_DEATHS_100K[country].max()
    real = 0.98 * RECOVERED_DEATHS_100K[country] / max_country
    forecast, p, q = compute_bass_model(real)
    bass_data[country] = (p, q)
    forecast = [w * max_country for w in forecast]
    forecast = pd.DataFrame(
        {"real": RECOVERED_DEATHS_100K[country], "forecast": forecast},
        index=RECOVERED_DEATHS_100K.index,
    )
    #
    BASS_RECOVERED_DEATHS_100K[country] = forecast.forecast
    #
    plt.plot(forecast.real, "-k", linewidth=2, alpha=1.0)
    plt.plot(forecast.forecast, "-r", linewidth=5, alpha=0.4)
    plt.text(0, 0.60 * plt.ylim()[1], "p={:5.4f}".format(p))
    plt.text(0, 0.50 * plt.ylim()[1], "q={:5.4f}".format(q))
    format_plot(country)

In [ ]:
BASS_ACTIVE_100K = BASS_CONFIRMED_100K - BASS_RECOVERED_DEATHS_100K

plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    plt.plot(ACTIVE_100K[country], c='k', alpha=0.5)
    plt.plot(BASS_ACTIVE_100K[country], '-r', linewidth=3)
    format_plot(country)

## Tasa congatio

In [ ]:
d = BASS_ACTIVE_100K.shift(periods=1, fill_value=1)
d.applymap(lambda w: 1 if w == 0 else w)
d.applymap(lambda w: 1 if pd.isna(w) else w)
BASS_TASA_CONTAGIO_100K = BASS_NEW_CASES_100K / d

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(4, 5, i + 1)
    plt.plot(TASA_CONTAGIO[country], c='k', alpha=0.5)
    plt.plot(BASS_TASA_CONTAGIO_100K[country], '-r', linewidth=3)
    format_plot(country)

# INDICADORES DE CARACTERIZACION BASS

In [ ]:
bass = pd.DataFrame()
bass["confirmed_100k"] = list(BASS_CONFIRMED_100K.max())
bass.index = BASS_CONFIRMED_100K.columns

#
# Máximo cantidad de casos activos (pico de la curva)
#
bass["max_active_100k"] = 0.0
for country in countries:
    bass.at[country, "max_active_100k"] = BASS_ACTIVE_100K[country].max()
    
##
## Máximo número de casos diario para t >= 100 casos totales
##
bass["max_new_cases_100k"] = 0.0
for country in countries:
    bass.at[country, "max_new_cases_100k"] = BASS_NEW_CASES_100K[country][
        CONFIRMED[country] > 100
    ].max()
    
#
# Maxima tasa diaria de contagio para
# para t >= 100 casos totales
#
bass["max_tasa_contagio"] = 0.0
for country in countries:
    bass.at[country, "max_tasa_contagio"] = BASS_TASA_CONTAGIO_100K[country][
        CONFIRMED[country] > 100
    ].max()
    
#
# Dias al pico de activos
#
bass["days_to_max_active"] = 0
BASS_ACTIVE_100K["days"] = range(len(ACTIVE_100K))
for country in countries:
    first_day = sum(CONFIRMED[country].map(lambda x: x < 100))
    pike_day = int(
        BASS_ACTIVE_100K["days"][
            BASS_ACTIVE_100K[country].map(lambda x: x == BASS_ACTIVE_100K[country].max())
        ].head(1)
    )
    bass.at[country, "days_to_max_active"] = pike_day - first_day

BASS_ACTIVE_100K.pop("days")



#
# Dias entre el 20% y 80% infectados
#
BASS_CONFIRMED_100K["days"] = range(len(CONFIRMED))
bass["days_20_to_80"] = 0

for country in countries:
    first_day = sum(
        BASS_CONFIRMED_100K[country].map(lambda x: x < 0.20 * BASS_CONFIRMED_100K[country].max())
    )
    last_day = sum(
        BASS_CONFIRMED_100K[country].map(lambda x: x < 0.80 * BASS_CONFIRMED_100K[country].max())
    )
    bass.at[country, "days_20_to_80"] = last_day - first_day

BASS_CONFIRMED_100K.pop("days")
bass

In [ ]:
bass.to_csv('bass.csv')

In [ ]:
world_data = world_data.set_index('country')

In [ ]:
world_data['Confirmed'] = CONFIRMED.max()
world_data['Active'] = ACTIVE.loc[ACTIVE.index[-1],:]
world_data['Active/Confirmed'] = world_data['Active'] / world_data['Confirmed']

In [ ]:
    
plt.figure(figsize=FIGSIZE)
for i, country in enumerate(countries):
    plt.subplot(5, 4, i + 1)
    max_country = CONFIRMED_100K[country].max()
    p, q = bass_data[country]
    yt = CONFIRMED_100K[country][CONFIRMED_100K[country] > 0].tolist()
    bass = bass_model_ft(len(yt), p, q)
    bass = [w * max_country for w in bass]
    bass = [0] * (len(CONFIRMED_100K) - len(yt)) + list(bass)
    forecast = pd.DataFrame(
        {"real": ACTIVE_100K[country], "forecast": bass},
        index=ACTIVE_100K.index,
    )
    plt.plot(forecast.real, "-k", linewidth=2)
    plt.plot(forecast.forecast, "-r", linewidth=3, alpha=0.5)
    plt.text(0, 0.60 * plt.ylim()[1], "p={:5.4f}".format(p))
    plt.text(0, 0.50 * plt.ylim()[1], "q={:5.4f}".format(q))
    format_plot(country)    

# Modelo optimizacion tasa de contagio

In [ ]:
#  def analyze_adaptive_rate(LAMBDA, min_pop):
#      def compute_rate(country, T):
#          def loss(rate):
#              rate_obs = TASA_CONTAGIO[country][T]
#              e2 = sum([abs(r - o) ** 2 for r, o in zip(rate_obs, rate)])
#              last_terms = sum(
#                  [abs(r - o) ** 2 for r, o in zip(rate_obs[-5:], rate[-5:])]
#              )
#              pen0 = sum([abs(max(0, v - u)) for u, v in zip(rate[:-1], rate[1:])])
#              pen1 = sum([abs(min(0, u)) for u in rate])
#              return LAMBDA * e2 + (1 - LAMBDA) * pen0 + 1 * pen1 + 5 * last_terms
#
#          w = TASA_CONTAGIO[country][T].tolist()
#          w[0] = max(w)
#          w = minimize(loss, w).x
#          return w
#
#      tasa_ajustada = TASA_CONTAGIO.applymap(lambda x: None)
#      for country in countries:
#          print("*", sep="", end="")
#          T = list(CONFIRMED[CONFIRMED[country] > min_pop][country].index)
#          tasa0 = compute_rate(country, T)
#          for index, t in enumerate(T):
#              tasa_ajustada.at[t, country] = tasa0[index]
#          # break
#
#      plt.figure(figsize=(FIGSIZE))
#      for i, country in enumerate(countries):
#          plt.subplot(5, 4, i + 1)
#          plt.plot(TASA_CONTAGIO[country], alpha=0.5)
#          plt.plot(tasa_ajustada[country], color="r", linewidth=2)
#          format_plot(country)
#          # break

In [ ]:
#  analyze_adaptive_rate(LAMBDA=0.70, min_pop=100)

In [ ]:
#  analyze_adaptive_rate(LAMBDA=0.65, min_pop=100)

In [ ]:
#  analyze_adaptive_rate(LAMBDA=0.60, min_pop=100)

In [ ]:
#  population['mauritius']

# Modelo SIR

In [ ]:
#  FIGSIZE = (15, 18)

In [ ]:
#  def compute_adaptive_rate(LAMBDA):
#      def compute_rate(country, T):
#          def loss(rate):
#              rate_obs = tasa[country][T]
#              e2 = sum([abs(r - o) ** 2 for r, o in zip(rate_obs, rate)])
#              last_terms = sum(
#                  [abs(r - o) ** 2 for r, o in zip(rate_obs[-5:], rate[-5:])]
#              )
#              pen0 = sum([abs(max(0, v - u)) for u, v in zip(rate[:-1], rate[1:])])
#              pen1 = sum([abs(min(0, u)) for u in rate])
#              return LAMBDA * e2 + (1 - LAMBDA) * pen0 + 1 * pen1 + 5 * last_terms
#
#          w = tasa[country][T].tolist()
#          w[0] = max(w)
#          w = minimize(loss, w).x
#          return w
#
#      tasa_ajustada = tasa.applymap(lambda x: None)
#      for country in countries:
#          print("*", sep="", end="")
#          T = list(CONFIRMED[CONFIRMED[country] > 10][country].index)
#          tasa0 = compute_rate(country, T)
#          for index, t in enumerate(T):
#              tasa_ajustada.at[t, country] = tasa0[index]
#
#      # plt.figure(figsize=(FIGSIZE))
#      #  for i, country in enumerate(countries):
#      #      plt.subplot(5, 4, i + 1)
#      #      plt.plot(tasa[country], alpha=0.5)
#      #      plt.plot(tasa_ajustada[country], color='r', linewidth = 2)
#      #      format_plot(country)
#
#      return tasa_ajustada
#
#
#  tasa_pronosticada = compute_adaptive_rate(LAMBDA=0.8)

In [ ]:
#  from numpy import power, prod
#
#  def media_tasas(x):
#      n = len(x)
#      x = [u + 1 for u in x]
#      x = prod(x)
#      x = power(x, 1.0 / n) - 1.0
#      return x

In [ ]:
#  sim_total_casos = pd.concat([tasa] * 4).applymap(lambda x: None)
#  sim_total_casos = sim_total_casos.reset_index(drop=True)
#
#  sim_total_recuperados = sim_total_casos.copy()
#  sim_total_muertes = sim_total_casos.copy()
#
#  sim_casos_activos = sim_total_casos.copy()
#  sim_contagios = sim_total_casos.copy()
#  sim_recuperados = sim_total_casos.copy()
#  sim_muertes = sim_total_casos.copy()
#
#
#  for country in countries:
#
#      T = list(CONFIRMED[CONFIRMED[country] > 1][country].index)
#
#      sim_tasa_fallecimiento = tasa_fallecimiento[country]
#      sim_tasa_fallecimiento = sim_tasa_fallecimiento[
#          list(sim_tasa_fallecimiento.index)[-100:]
#      ]
#      sim_tasa_fallecimiento = media_tasas(
#          sim_tasa_fallecimiento[sim_tasa_fallecimiento > 0]
#      )
#
#      sim_tasa_recuperacion = tasa_recuperacion[country]
#      sim_tasa_recuperacion = sim_tasa_recuperacion[
#          list(sim_tasa_recuperacion.index)[-100:]
#      ]
#      sim_tasa_recuperacion = media_tasas(
#          sim_tasa_recuperacion[sim_tasa_recuperacion > 0]
#      )
#
#      for t in T:
#
#          sim_total_casos.at[t, country] = CONFIRMED[country][t]
#          sim_total_recuperados.at[t, country] = RECOVERED[country][t]
#          sim_total_muertes.at[t, country] = DEATHS[country][t]
#
#          sim_casos_activos.at[t, country] = infectados[country][t]
#          sim_contagios.at[t, country] = contagios[country][t]
#          if t == 0:
#              sim_muertes.at[t, country] = DEATHS[country][t]
#          else:
#              sim_muertes.at[t, country] = DEATHS[country][t] - DEATHS[country][t - 1]
#
#          if t == 0:
#              sim_recuperados.at[t, country] = RECOVERED[country][t]
#          else:
#              sim_recuperados.at[t, country] = (
#                  RECOVERED[country][t] - RECOVERED[country][t - 1]
#              )
#
#      #  max_pop_affected = population[country] * 0.015
#
#      max_pop_affected = 1500
#
#      tasa_pron = tasa_pronosticada[country][len(tasa_pronosticada) - 1]
#
#      for t in range(T[-1], len(sim_total_casos)):
#
#          sim_contagios.at[t, country] = (
#              tasa_pron
#              * sim_casos_activos.at[t - 1, country]
#              * (max_pop_affected - sim_total_casos[country][t - 1])
#              / max_pop_affected
#          )
#
#          sim_muertes.at[t, country] = (
#              sim_tasa_fallecimiento * sim_casos_activos.at[t - 1, country]
#          )
#
#          sim_recuperados.at[t, country] = (
#              sim_tasa_recuperacion * sim_casos_activos.at[t - 1, country]
#          )
#
#          sim_casos_activos.at[t, country] = (
#              sim_casos_activos.at[t - 1, country]
#              + sim_contagios.at[t, country]
#              - sim_muertes.at[t, country]
#              - sim_recuperados.at[t, country]
#          )
#
#          sim_total_casos.at[t, country] = (
#              sim_total_casos.at[t - 1, country] + sim_contagios[country][t]
#          )
#          sim_total_recuperados.at[t, country] = (
#              sim_total_recuperados.at[t - 1, country] + sim_recuperados[country][t]
#          )
#          sim_total_muertes.at[t, country] = (
#              sim_total_muertes[country][t - 1] + sim_muertes[country][t]
#          )

In [ ]:
#  plt.figure(figsize=(14, 60))
#
#  for i, country in enumerate(countries):
#      plt.subplot(20, 4, 4 * i + 1)
#      plt.plot(tasa[country], alpha=0.5)
#      plt.plot(tasa_pronosticada[country], color="r", linewidth=2)
#      format_plot(country)
#
#      plt.subplot(20, 4, 4 * i + 2)
#      plt.plot(tasa_recuperacion[country], alpha=0.5)
#      format_plot(country)
#
#      plt.subplot(20, 4, 4 * i + 3)
#      plt.plot(sim_total_casos[country], "--k")
#      plt.plot(CONFIRMED[country], "-k", linewidth=2)
#      plt.plot(sim_total_recuperados[country], "--g")
#      plt.plot(RECOVERED[country], "-g")
#      format_plot(country)
#
#      plt.subplot(20, 4, 4 * i + 4)
#      plt.plot(sim_casos_activos[country], "-r")
#      plt.plot(infectados[country], "-k")
#      format_plot(country)